<a href="https://colab.research.google.com/github/danieligelnik/CCFraudProject/blob/main/Test_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies as needed:
%pip install kagglehub[pandas-datasets]

In [ ]:
!pip install import-ipynb
import import_ipynb
import requests
import os

# 1. URL to the RAW version of the notebook on GitHub
github_url = "https://raw.githubusercontent.com/danieligelnik/CCFraudProject/main/help_functions.ipynb"
# Corrected the typo in the filename here:
notebook_filename = "help_functions.ipynb"

# 2. Download the notebook file locally
response = requests.get(github_url)
with open(notebook_filename, 'wb') as f:
    f.write(response.content)

# 3. Import the notebook as a module
try:
    # The module name must match the filename (without .ipynb)
    from help_functions import *
    #from help_functions import load_kagglehub_dataset
    print(f"Successfully imported functions from {notebook_filename}")
except Exception as e:
    print(f"Error importing notebook: {e}")

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import holidays as hol
from sklearn.model_selection import train_test_split, KFold

# **Data**

### Download data set & removed is_fraud category

**Data encoding**

**Feature engineering**

In [ ]:
df_cards_test_part = load_kagglehub_dataset("dermisfit/fraud-transactions-dataset", 'fraudTest.csv')
df_cards_test_eng = feature_engineering(df_cards_test_part)
df_info(df_cards_test_eng,'head')

**Dropping redundant features**

In [ ]:
df_test_eng = df_cards_test_eng.drop(columns=['trans_date_trans_time','Unnamed: 0','merchant','long','job','trans_num','unix_time','merch_long', 'dob', 'city', 'state','first','last','street'])

# **Tesing XGBoost on test part with all tuning from the training**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Separate features (X) and target (y) for training data
X_test = df_test_eng.drop('is_fraud', axis=1)
y_test = df_test_eng['is_fraud']

# Calculate scale_pos_weight for XGBoost (sum of negative instances / sum of positive instances)
# This is the standard way XGBoost handles class imbalance
negative_cases = np.sum(y_test == 0)
positive_cases = np.sum(y_test == 1)
scale_weight = negative_cases / positive_cases

# Initialize XGBoost Classifier
xgb_model = XGBClassifier(
    n_estimators=100,
    subsample=0.8,
    learning_rate=0.1,
    max_depth=3,
    scale_pos_weight=scale_weight,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

print("Training XGBoost model... This might take a while.")
scaler = StandardScaler()
X_test_scaled = scaler.fit_transform(X_test)

# Train the model
xgb_model.fit(X_test_scaled, y_test)

print("\nXGBoost model trained.")

# Make predictions with probability threshold 0.4
y_prob = xgb_model.predict_proba(X_test_scaled)[:, 1]
threshold = 0.4
y_pred = (y_prob >= threshold).astype(int)

# Create result table with one row and metrics as columns
result_ft = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1-Score': f1_score(y_test, y_pred),
    'ROC AUC': roc_auc_score(y_test, y_prob)
}

display(pd.DataFrame([result_ft]))

# Display Confusion Matrix as DataFrame
conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
display(pd.DataFrame(conf_matrix,
                         index=['Actual Normal', 'Actual Fraud'],
                         columns=['Predicted Normal', 'Predicted Fraud']))

# Display Confusion Matrix Plot
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Normal', 'Predicted Fraud'],
            yticklabels=['Actual Normal', 'Actual Fraud'])
plt.title(f'Confusion Matrix Heatmap (Threshold = {threshold})')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()